# ARC-05: Orbital Propagation Engine

This notebook combines fixed-step RK4 with the point-mass two-body derivative. A final partial timestep is shortened so propagation ends exactly at the configured duration.

Collision times and state are radial-linear estimates for visualization and validation only. They are not physically exact impact solutions.

Trajectory data uses `times_s` and a Float64 state matrix with columns `[x, y, z, vx, vy, vz]`. Units are SI and the frame is ECI.


In [1]:
import math
import numpy as np


def _stable_position_radius(position):
    return float(np.hypot(np.hypot(abs(position[0]), abs(position[1])), abs(position[2])))


def _numeric_state(state):
    if isinstance(state, (str, bytes, bytearray, bool, np.bool_)):
        raise ValueError("State must be numeric")
    try:
        components = list(state)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State must be numeric") from exc
    for component in components:
        if isinstance(component, (bool, np.bool_, str, bytes, bytearray, complex, np.complexfloating, list, tuple, np.ndarray)):
            raise ValueError("State must contain only real numeric values")
    y = np.asarray(state, dtype=np.float64)
    if y.shape != (6,) or not np.all(np.isfinite(y)):
        raise ValueError("State must be a finite six-component vector")
    return y


def rk4_step(state, time_s, time_step_s, derivative, *args):
    if isinstance(time_step_s, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Timestep must be numeric")
    dt = float(time_step_s)
    t = float(time_s)
    if not math.isfinite(dt) or dt <= 0:
        raise ValueError("Timestep must be finite and positive")
    if not math.isfinite(t) or t < 0:
        raise ValueError("Time must be finite and non-negative")
    y = _numeric_state(state)
    k1 = np.asarray(derivative(t, y, *args), dtype=np.float64)
    k2 = np.asarray(derivative(t + 0.5*dt, y + 0.5*dt*k1, *args), dtype=np.float64)
    k3 = np.asarray(derivative(t + 0.5*dt, y + 0.5*dt*k2, *args), dtype=np.float64)
    k4 = np.asarray(derivative(t + dt, y + dt*k3, *args), dtype=np.float64)
    for k in (k1, k2, k3, k4):
        if k.shape != y.shape or not np.all(np.isfinite(k)):
            raise ValueError("RK4 intermediate stage is invalid")
    result = y + dt/6.0*(k1 + 2*k2 + 2*k3 + k4)
    if not np.all(np.isfinite(result)):
        raise ValueError("RK4 produced a non-finite state")
    return result


def two_body_derivative(time_s, state, mu):
    y = _numeric_state(state)
    if isinstance(mu, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Gravitational parameter must be numeric")
    mu = float(mu)
    if not math.isfinite(mu) or mu <= 0:
        raise ValueError("Gravitational parameter must be finite and positive")
    radius = _stable_position_radius(y[:3])
    if not math.isfinite(radius) or radius <= 0:
        raise ValueError("Position radius must be finite and positive")
    position_unit = y[:3] / radius
    acceleration_scale = (mu / radius) / radius
    if not math.isfinite(acceleration_scale):
        raise ValueError("Acceleration magnitude is non-finite")
    if acceleration_scale == 0.0:
        raise ValueError("Acceleration magnitude underflows float64")
    acceleration = -acceleration_scale * position_unit
    result = np.concatenate((y[3:], acceleration)).astype(np.float64)
    if not np.all(np.isfinite(result)):
        raise ValueError("Derivative contains non-finite values")
    return result


def interpolate_collision_event(previous_time_s, previous_state, next_time_s, next_state, collision_radius_m):
    previous = _numeric_state(previous_state)
    next_y = _numeric_state(next_state)
    previous_radius = _stable_position_radius(previous[:3])
    next_radius = _stable_position_radius(next_y[:3])
    if not math.isfinite(previous_radius) or not math.isfinite(next_radius):
        raise ValueError("Collision bracket contains a non-finite radius")
    if isinstance(previous_time_s, (bool, np.bool_, str, bytes, bytearray)) or isinstance(next_time_s, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Collision bracket times must be finite numeric scalars")
    try:
        previous_time = float(previous_time_s)
        next_time = float(next_time_s)
        threshold = float(collision_radius_m)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("Collision bracket times and threshold must be numeric") from exc
    if not math.isfinite(previous_time) or not math.isfinite(next_time) or next_time < previous_time:
        raise ValueError("Collision bracket times must be finite and ordered")
    if isinstance(collision_radius_m, (bool, np.bool_, str, bytes, bytearray)) or not math.isfinite(threshold) or threshold <= 0:
        raise ValueError("Collision threshold must be finite and positive")
    if previous_radius == threshold:
        return previous_time, previous.copy(), "exact_sample"
    if next_radius == threshold:
        return next_time, next_y.copy(), "exact_sample"
    if previous_radius == next_radius:
        raise ValueError("Invalid collision interval: endpoint radii are degenerate and do not bracket the threshold")
    if not (min(previous_radius, next_radius) <= threshold <= max(previous_radius, next_radius)):
        raise ValueError("Invalid collision interval: threshold is not bracketed by endpoint radii")
    radius_difference = next_radius - previous_radius
    alpha = (threshold - previous_radius) / radius_difference
    if not math.isfinite(alpha):
        raise ValueError("Invalid collision interval: interpolation fraction is non-finite")
    if alpha < 0.0 or alpha > 1.0:
        raise ValueError("Invalid collision interval: interpolation fraction is outside [0, 1]")
    event_time = previous_time + alpha*(next_time-previous_time)
    event_state = (1.0-alpha)*previous + alpha*next_y
    if not np.all(np.isfinite(event_state)):
        raise ValueError("Interpolated collision state is non-finite")
    return event_time, event_state.astype(np.float64), "linear_radius_interpolation"


def propagate_two_body(initial_state, mu, duration_s, time_step_s, collision_radius_m):
    y0 = _numeric_state(initial_state)
    for name, value in (("mu", mu), ("duration", duration_s), ("timestep", time_step_s), ("collision radius", collision_radius_m)):
        if isinstance(value, (bool, np.bool_, str, bytes, bytearray)):
            raise ValueError(f"{name} must be numeric")
    mu = float(mu)
    duration = float(duration_s)
    timestep = float(time_step_s)
    collision_radius = float(collision_radius_m)
    if not math.isfinite(mu) or mu <= 0:
        raise ValueError("Gravitational parameter must be finite and positive")
    if not math.isfinite(duration) or duration <= 0:
        raise ValueError("Simulation duration must be finite and positive")
    if not math.isfinite(timestep) or timestep <= 0:
        raise ValueError("Timestep must be finite and positive")
    if not math.isfinite(collision_radius) or collision_radius <= 0:
        raise ValueError("Collision radius must be finite and positive")
    initial_radius = float(np.linalg.norm(y0[:3]))
    if not math.isfinite(initial_radius) or initial_radius <= 0:
        raise ValueError("Initial position radius must be finite and positive")

    times = [0.0]
    states = [y0.copy()]
    collision_index = None
    collision_time_s = None
    collision_state = None
    collision_method = None
    current_time = 0.0
    y = y0.copy()
    if float(np.linalg.norm(y[:3])) <= collision_radius:
        collision_index = 0
        collision_time_s = 0.0
        collision_state = y.copy()
        collision_method = "exact_sample"
    while current_time < duration and collision_index is None:
        step = min(timestep, duration-current_time)
        previous_time = current_time
        previous_state = y.copy()
        y = rk4_step(y, current_time, step, two_body_derivative, mu)
        current_time += step
        if abs(current_time-duration) <= 1e-12*max(1.0, duration):
            current_time = duration
        times.append(current_time)
        states.append(y.copy())
        if float(np.linalg.norm(y[:3])) <= collision_radius:
            collision_index = len(states)-1
            collision_time_s, collision_state, collision_method = interpolate_collision_event(
                previous_time, previous_state, current_time, y, collision_radius
            )
    return {
        "times_s": np.asarray(times, dtype=np.float64),
        "states": np.vstack(states).astype(np.float64),
        "collision_index": collision_index,
        "collision_radius_m": collision_radius,
        "collision_time_s": collision_time_s,
        "collision_state": None if collision_state is None else collision_state.astype(np.float64),
        "collision_method": collision_method,
    }


In [2]:
mu = 3.986004418e14
radius = 7_000_000.0
circular_speed = math.sqrt(mu/radius)
initial_state = np.array([radius, 0.0, 0.0, 0.0, circular_speed, 0.0], dtype=np.float64)
trajectory = propagate_two_body(initial_state, mu, 600.0, 10.0, 6_378_137.0)
assert trajectory["times_s"][0] == 0.0
assert trajectory["times_s"][-1] == 600.0
assert np.all(np.diff(trajectory["times_s"]) > 0.0)
assert trajectory["states"].shape == (61, 6)
assert trajectory["collision_index"] is None
partial = propagate_two_body(initial_state, mu, 25.0, 10.0, 6_378_137.0)
assert np.allclose(partial["times_s"], np.array([0.0, 10.0, 20.0, 25.0]))
assert partial["times_s"][-1] == 25.0
collision = propagate_two_body(np.array([6_378_137.0, 0, 0, 0, 0, 0], dtype=np.float64), mu, 100.0, 1.0, 6_378_137.0)
assert collision["collision_index"] == 0
for invalid in ([1, 2, 3], [0, 0, 0, 0, 0, 0], [1, 2, 3, 4, 5, float("nan")]):
    try:
        propagate_two_body(invalid, mu, 1.0, 0.1, 6_378_137.0)
        raise AssertionError("Invalid initial state accepted")
    except ValueError:
        pass
print("Propagation trajectory shape:", trajectory["states"].shape)
print("Partial timestep policy: PASSED")
print("Initial collision detection: PASSED")


Propagation trajectory shape: (61, 6)
Partial timestep policy: PASSED
Initial collision detection: PASSED
